# Milestone 2: Data Quality dan Profiling

Lanjutan `01_data_profiling.ipynb`. Notebook ini menerapkan profiling DuckDB `SUMMARIZE` dan transformasi Polars
untuk missing values, outlier, duplikat, dan validasi pada dataset `akhyar016/dgph-covid19-social-media`
(komentar/submission Reddit bertema COVID-19).

**Sumber data:** secara default memakai sampel `data/preview/dgph_preview_sample.parquet` (200 baris, ikut di-commit).
Bila shard parquet penuh sudah diunduh lokal ke `data/raw/*.parquet`, notebook otomatis memakainya
(skema mentah dengan kolom `content` juga ditangani). Seluruh aturan cleaning ditulis sebagai ekspresi Polars
sehingga identik untuk sampel maupun shard.

## 0. Konfigurasi dan pemuatan data

In [1]:
from pathlib import Path

import duckdb
import polars as pl

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

RAW_FILES = sorted((ROOT / "data" / "raw").rglob("*.parquet"))
PREVIEW_PATH = ROOT / "data" / "preview" / "dgph_preview_sample.parquet"
DATA_PATHS = RAW_FILES or [PREVIEW_PATH]
OUTPUT_PATH = ROOT / "data" / "processed" / "dgph_bersih.parquet"

if not DATA_PATHS[0].exists():
    raise FileNotFoundError(f"Data tidak ditemukan: {DATA_PATHS[0]}. Jalankan 01_data_profiling.ipynb dahulu.")

print("Sumber:", "shard penuh" if RAW_FILES else "sampel preview", f"({len(DATA_PATHS)} file)")

Sumber: sampel preview (1 file)


In [2]:
lf = pl.scan_parquet(DATA_PATHS)
cols = lf.collect_schema().names()

# Skema mentah HF menyimpan teks di list<struct{text,title}> pada kolom `content`;
# skema sampel sudah diratakan menjadi `text` dan `title`.
if "content" in cols:
    lf = lf.with_columns(
        pl.col("content").list.first().struct.field("text").alias("text"),
        pl.col("content").list.first().struct.field("title").alias("title"),
    ).drop("content")

# Ambil field berguna dari metadata (string JSON berisi list satu elemen)
def meta_int(path: str, name: str) -> pl.Expr:
    return pl.col("metadata").str.json_path_match(path).cast(pl.Int64, strict=False).alias(name)

lf = lf.with_columns(
    meta_int("$[0].score", "score"),
    meta_int("$[0].num_comments", "num_comments"),
    pl.col("metadata").str.json_path_match("$[0].subreddit").alias("subreddit"),
).drop("metadata")

df = lf.collect()
print(f"Ukuran awal: {df.shape[0]:,} baris x {df.shape[1]} kolom")
df.head()

Ukuran awal: 10,000 baris x 12 kolom


id,title,text,timestamp,source,url,parent_id,content_type,submission_id,score,num_comments,subreddit
str,str,str,"datetime[μs, UTC]",str,str,str,str,str,i64,i64,str
"""reddit_t3_fcvf8o""","""Healthcare professionals: what…","""Hi all. I've been following C…",2020-03-03 14:35:23 UTC,"""reddit_coronavirusga""","""/r/CoronavirusGA/comments/fcvf…","""reddit_t3_fcvf8o""","""reddit_submission""","""reddit_t3_fcvf8o""",6,3,"""CoronavirusGA"""
"""reddit_fjdc5lm""","""Healthcare professionals: what…","""According to the WHO China joi…",2020-03-03 15:44:35 UTC,"""reddit_coronavirusga""","""/r/CoronavirusGA/comments/fcvf…","""comment_of_reddit_fjdbc0p""","""reddit_comment""","""reddit_t3_fcvf8o""",5,0,"""CoronavirusGA"""
"""reddit_fjgaig9""","""Healthcare professionals: what…","""No. Obviously the government r…",2020-03-04 10:57:15 UTC,"""reddit_coronavirusga""","""/r/CoronavirusGA/comments/fcvf…","""comment_of_reddit_fjfu96m""","""reddit_comment""","""reddit_t3_fcvf8o""",1,0,"""CoronavirusGA"""
"""reddit_fjfu96m""","""Healthcare professionals: what…","""In your estimate, are the poli…",2020-03-04 05:40:37 UTC,"""reddit_coronavirusga""","""/r/CoronavirusGA/comments/fcvf…","""comment_of_reddit_fjfg5kn""","""reddit_comment""","""reddit_t3_fcvf8o""",1,1,"""CoronavirusGA"""
"""reddit_fjfg5kn""","""Healthcare professionals: what…","""You’re really responsible for …",2020-03-04 03:05:20 UTC,"""reddit_coronavirusga""","""/r/CoronavirusGA/comments/fcvf…","""comment_of_reddit_fjf9ub9""","""reddit_comment""","""reddit_t3_fcvf8o""",-1,1,"""CoronavirusGA"""


## 1. Profiling dengan DuckDB `SUMMARIZE`

Ringkasan tipe data, null%, min/max, dan jumlah nilai distinct per kolom dalam satu perintah.

In [3]:
connection = duckdb.connect()
connection.register("dataset", df)

profil = connection.sql(
    "SUMMARIZE SELECT id, title, text, timestamp, source, content_type, score, num_comments, subreddit FROM dataset"
).pl()
profil.select("column_name", "column_type", "min", "max", "approx_unique", "avg", "null_percentage")

column_name,column_type,min,max,approx_unique,avg,null_percentage
str,str,str,str,i64,str,"decimal[9,2]"
"""id""","""VARCHAR""","""reddit_ffuc4wg""","""reddit_t3_z13c3x""",9785,null,0.00
"""title""","""VARCHAR""","""""I'd be so embarrassed if I go…","""“Negative covid” but mysteriou…",485,null,0.00
"""text""","""VARCHAR""",""" Cool, wenn der Impfstoff gut…","""🤦🏻‍♂️""",9928,null,0.00
"""timestamp""","""TIMESTAMP WITH TIME ZONE""","""2020-01-29 02:14:23+07""","""2022-12-16 07:06:46+07""",9586,"""2021-04-08 00:51:11.3562+07""",0.00
"""source""","""VARCHAR""","""reddit_anchorage""","""reddit_wisconsin""",67,null,0.00
"""content_type""","""VARCHAR""","""reddit_comment""","""reddit_submission""",2,null,0.00
"""score""","""BIGINT""","""-55""","""1661""",170,"""6.517""",0.00
"""num_comments""","""BIGINT""","""0""","""156""",39,"""0.8825""",0.00
"""subreddit""","""VARCHAR""","""Atlanta""","""wisconsin""",67,null,0.00


## 2. Deteksi Missing Values

Selain `null`, teks Reddit memiliki penanda hapus (`[removed]`, `[deleted]`) dan string kosong yang secara semantik juga "hilang".

In [4]:
jumlah_baris = len(df)
null_summary = (
    df.null_count()
    .transpose(include_header=True, header_name="kolom", column_names=["jumlah_null"])
    .with_columns((pl.col("jumlah_null") / jumlah_baris * 100).round(2).alias("persen_null"))
    .sort("jumlah_null", descending=True)
)
print("Null per kolom:")
display(null_summary.filter(pl.col("jumlah_null") > 0) if null_summary["jumlah_null"].sum() else null_summary.head(3))

PENANDA_HAPUS = ["[removed]", "[deleted]"]
teks_hilang = pl.col("text").is_null() | (pl.col("text").str.strip_chars() == "") | pl.col("text").is_in(PENANDA_HAPUS)
print(f"Teks null/kosong/dihapus: {df.filter(teks_hilang).height:,} baris ({df.filter(teks_hilang).height / jumlah_baris:.1%})")
df.filter(teks_hilang).group_by("text").len().sort("len", descending=True)

Null per kolom:


kolom,jumlah_null,persen_null
str,u32,f64
"""id""",0,0.0
"""title""",0,0.0
"""text""",0,0.0


Teks null/kosong/dihapus: 235 baris (2.4%)


text,len
str,u32
"""[deleted]""",139
"""[removed]""",96


## 3. Penanganan Missing Values

Strategi (disesuaikan dengan jenis kolom):

| Kolom | Masalah | Strategi | Alasan |
|---|---|---|---|
| `text` | null / kosong / `[removed]` / `[deleted]` | **Hapus baris** | Tanpa teks, baris tidak bermakna untuk analisis teks/topik; mengisi nilai akan menyesatkan. |
| `title` | null | Isi `""` | Komentar tidak selalu punya judul sendiri; kosong adalah nilai valid. |
| `score`, `num_comments` | null / gagal diparse | Isi **median** | Numerik berdistribusi miring; median tahan terhadap pencilan. |
| `subreddit` | null | Isi dari `source` (hilangkan prefiks `reddit_`) | Informasi redundan tersedia di kolom lain. |

In [5]:
df_missing_handled = (
    df.filter(~teks_hilang)
    .with_columns(
        pl.col("title").fill_null(""),
        pl.col("score").fill_null(pl.col("score").median()),
        pl.col("num_comments").fill_null(pl.col("num_comments").median()),
        pl.col("subreddit").fill_null(pl.col("source").str.replace("^reddit_", "")),
    )
)

print(f"Baris dihapus karena teks hilang: {len(df) - len(df_missing_handled):,}")
print(f"Null sebelum: {df.null_count().sum_horizontal().sum():,}")
print(f"Null sesudah: {df_missing_handled.null_count().sum_horizontal().sum():,}")

Baris dihapus karena teks hilang: 235
Null sebelum: 0
Null sesudah: 0


## 4. Deteksi dan Penanganan Outlier

Dua variabel numerik utama: `score` (jumlah upvote bersih, bisa negatif) dan panjang teks (`panjang_teks`).
Metode: IQR (1,5 × IQR) sebagai penanda statistik, dipadukan dengan batas domain:

- `score`: **winsorization** ke batas IQR pada kolom baru `score_clip` (nilai asli `score` tetap disimpan). Skor viral yang sangat besar valid, tetapi mendistorsi rata-rata.
- `panjang_teks`: **tidak dipotong** (teks tidak boleh dimodifikasi); baris ditandai `outlier_panjang` agar bisa difilter saat analisis. Teks sangat pendek (< 3 karakter) tetap dianggap tidak informatif dan dibuang.

In [6]:
KOLOM_NUMERIK = "score"
MIN_PANJANG = 3

df_len = df_missing_handled.with_columns(pl.col("text").str.len_chars().alias("panjang_teks"))

def batas_iqr(s: pl.Series) -> tuple[float, float]:
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

bawah_s, atas_s = batas_iqr(df_len[KOLOM_NUMERIK])
bawah_p, atas_p = batas_iqr(df_len["panjang_teks"])
print(f"score        : batas [{bawah_s:.1f}, {atas_s:.1f}]  | di luar batas: {df_len.filter((pl.col('score') < bawah_s) | (pl.col('score') > atas_s)).height}")
print(f"panjang_teks : batas [{bawah_p:.1f}, {atas_p:.1f}] | di luar batas: {df_len.filter((pl.col('panjang_teks') < bawah_p) | (pl.col('panjang_teks') > atas_p)).height}")

df_outlier_handled = (
    df_len.filter(pl.col("panjang_teks") >= MIN_PANJANG)
    .with_columns(
        pl.col("score").clip(lower_bound=bawah_s, upper_bound=atas_s).alias("score_clip"),
        (pl.col("panjang_teks") > atas_p).alias("outlier_panjang"),
    )
)
print(f"Teks < {MIN_PANJANG} karakter dibuang: {len(df_len) - len(df_outlier_handled)}")
print(f"Ditandai outlier_panjang: {df_outlier_handled['outlier_panjang'].sum()}")

score        : batas [-6.5, 13.5]  | di luar batas: 1171
panjang_teks : batas [-310.5, 709.5] | di luar batas: 738
Teks < 3 karakter dibuang: 12
Ditandai outlier_panjang: 738


## 5. Deduplikasi

Kunci bisnis: `id` (unik per konten). Selain itu, komentar dengan teks identik pada thread yang sama (`submission_id`) dianggap salinan/spam dan hanya satu yang dipertahankan.

In [7]:
KOLOM_KUNCI = "id"

total_baris = len(df_outlier_handled)
baris_unik = df_outlier_handled[KOLOM_KUNCI].n_unique()
print(f"Total baris: {total_baris:,}")
print(f"Baris unik berdasarkan {KOLOM_KUNCI}: {baris_unik:,}")
print(f"Duplikat id: {total_baris - baris_unik:,}")

df_dedup = (
    df_outlier_handled
    .sort("timestamp")   # pertahankan kemunculan paling awal
    .unique(subset=[KOLOM_KUNCI], keep="first", maintain_order=True)
)
n_id = len(df_dedup)
df_dedup = df_dedup.unique(subset=["submission_id", "text"], keep="first", maintain_order=True)
print(f"Duplikat (submission_id, text) dibuang: {n_id - len(df_dedup)}")

Total baris: 9,753
Baris unik berdasarkan id: 9,753
Duplikat id: 0
Duplikat (submission_id, text) dibuang: 0


## 6. Validasi Data Dasar

Aturan kualitas dihitung dengan DuckDB; setiap pelanggaran harus bernilai 0 pada data bersih.

In [8]:
connection.register("dataset_bersih", df_dedup)

validasi = connection.sql("""
    SELECT
        COUNT(*)                                                                    AS total_baris,
        SUM(CASE WHEN id IS NULL OR text IS NULL THEN 1 ELSE 0 END)                 AS pelanggaran_null,
        SUM(CASE WHEN length(trim(text)) < 3 OR text IN ('[removed]','[deleted]')
                 THEN 1 ELSE 0 END)                                                 AS pelanggaran_teks,
        SUM(CASE WHEN timestamp < TIMESTAMPTZ '2019-12-01' OR timestamp >= TIMESTAMPTZ '2023-01-01'
                 THEN 1 ELSE 0 END)                                                 AS pelanggaran_periode,
        SUM(CASE WHEN content_type NOT IN ('reddit_submission','reddit_comment')
                 THEN 1 ELSE 0 END)                                                 AS pelanggaran_tipe,
        SUM(CASE WHEN NOT starts_with(id, 'reddit_') THEN 1 ELSE 0 END)             AS pelanggaran_prefiks_id,
        SUM(CASE WHEN score_clip < ? OR score_clip > ? THEN 1 ELSE 0 END)           AS pelanggaran_batas_score,
        COUNT(*) - COUNT(DISTINCT id)                                               AS pelanggaran_id_duplikat
    FROM dataset_bersih
""", params=[bawah_s, atas_s]).pl()
display(validasi)
assert validasi.drop("total_baris").row(0) == (0,) * (validasi.width - 1), "Ada pelanggaran aturan kualitas!"
print("Semua aturan validasi lulus.")

total_baris,pelanggaran_null,pelanggaran_teks,pelanggaran_periode,pelanggaran_tipe,pelanggaran_prefiks_id,pelanggaran_batas_score,pelanggaran_id_duplikat
i64,"decimal[38,0]","decimal[38,0]","decimal[38,0]","decimal[38,0]","decimal[38,0]","decimal[38,0]",i64
9753,0,0,0,0,0,0,0


Semua aturan validasi lulus.


## 7. Simpan Data Bersih

Disimpan ke `data/processed/` (di-ignore Git) untuk dipakai Milestone 3 tanpa mengulang cleaning.

In [9]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_dedup.write_parquet(OUTPUT_PATH)

print(f"Baris sebelum cleaning: {len(df):,}")
print(f"Baris sesudah cleaning: {len(df_dedup):,}  ({len(df_dedup) / len(df):.1%} dipertahankan)")
print(f"Data bersih tersimpan di: {OUTPUT_PATH.relative_to(ROOT)}")
df_dedup.head(3)

Baris sebelum cleaning: 10,000
Baris sesudah cleaning: 9,753  (97.5% dipertahankan)
Data bersih tersimpan di: data/processed/dgph_bersih.parquet


id,title,text,timestamp,source,url,parent_id,content_type,submission_id,score,num_comments,subreddit,panjang_teks,score_clip,outlier_panjang
str,str,str,"datetime[μs, UTC]",str,str,str,str,str,f64,f64,str,u32,f64,bool
"""reddit_ffufdt0""","""Roseville to Demolish Historic…","""I get that the church is beaut…",2020-01-28 19:46:36 UTC,"""reddit_detroit""","""/r/Detroit/comments/ev9sxu/ros…","""comment_of_reddit_t3_ev9sxu""","""reddit_comment""","""reddit_t3_ev9sxu""",5.0,1.0,"""Detroit""",124,5.0,false
"""reddit_ffuknl9""","""Roseville to Demolish Historic…","""hoarding""",2020-01-28 20:39:26 UTC,"""reddit_detroit""","""/r/Detroit/comments/ev9sxu/ros…","""comment_of_reddit_ffuburl""","""reddit_comment""","""reddit_t3_ev9sxu""",11.0,0.0,"""Detroit""",8,11.0,false
"""reddit_ffuspuo""","""Roseville to Demolish Historic…","""I can't believe that they are …",2020-01-28 21:57:11 UTC,"""reddit_detroit""","""/r/Detroit/comments/ev9sxu/ros…","""comment_of_reddit_t3_ev9sxu""","""reddit_comment""","""reddit_t3_ev9sxu""",10.0,2.0,"""Detroit""",278,10.0,false


## Catatan Milestone 2

**Hasil pada sampel 10.000 baris** (angka akan berbeda bila dijalankan pada shard penuh):

- **Dimensi kualitas paling bermasalah:** *validitas konten*, bukan null. Tidak ada `null` di kolom mana pun, tetapi 235 baris (2,4%) berisi penanda `[deleted]` (139) atau `[removed]` (96) sehingga tidak bermakna. Ini hanya terlihat dengan memeriksa nilai, bukan `null_count()`.
- **Missing values:** 235 baris dengan teks hilang dibuang; `title` kosong diisi `""`, numerik diisi median, `subreddit` diisi dari `source` (tidak ada yang terpakai pada sampel, aturan disiapkan untuk data penuh).
- **Outlier:** `score` bersifat miring ke kanan; 1.171 nilai (±12%) berada di luar batas IQR [-6,5; 13,5]. Diatasi dengan winsorization ke kolom `score_clip`, nilai asli `score` dipertahankan. `panjang_teks` memiliki 738 outlier (> 709,5 karakter) yang hanya **ditandai** (`outlier_panjang`), tidak dipotong. 12 teks < 3 karakter dibuang karena tidak informatif.
- **Duplikat:** 0 duplikat `id` dan 0 duplikat `(submission_id, text)` pada sampel. Pada data penuh (9,6 juta baris) hasilnya perlu dicek ulang.
- **Validasi (DuckDB):** 7 aturan (null, teks valid, periode 2019-12 s.d. 2022-12, `content_type`, prefiks `id`, batas `score_clip`, `id` unik), seluruhnya 0 pelanggaran.
- **Jumlah baris:** 10.000 → 9.753 (97,5% dipertahankan). Output: `data/processed/dgph_bersih.parquet` (15 kolom, di-ignore Git).
- **Keterbatasan:** sampel diambil dari 100 blok berurutan (±100 baris per blok) sehingga bukan sampel acak; batas IQR hanya indikatif. Untuk data penuh hitung ulang per shard atau dengan kuantil DuckDB (`approx_quantile`).